# Assignment 2

## Previous Assignment 

| DataNeeded                                                                                                                                                                                                                   | Source               | CollectionMethod            | DataType                                                          | StorageFormat |                                                                          |
|------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|----------------------|-----------------------------|-------------------------------------------------------------------|---------------|--------------------------------------------------------------------------|
| Movie List (Top 1000)                                                                                                                                                                                                        | IMDB.com             | Scraping                    | Qualitative (Title/IDs)                                           | CSV           |                                                                          |
| Movie Metadata                                                                                                                                                                                                               | OMDB API             | REST API Requests           | Qualitative (Genre) & Quantitative(Rating, Budget, Release Years) | JSON/CSV      |                                                                          |
| Inflation Data                                                                                                                                                                                                               | Kaggle.com           | CSV Download + Manual Entry | Quantitative(CPI indices)                                         | CSV           | <- numeric column that has some data with "." separator                  |
|                                                                                                                                                                                                                              | tradingeconomics.com |                             |                                                                   |               |                                                                          |
| Recession Years                                                                                                                                                                                                              | Wikipedia.org        | Manual Entry via Python     | Quantitative(Recession Years)                                     | CSV           | <- values specified in different formats (Yes/No and True/False and 0/1) |
|                                                                                                                                                                                                                              |                      |                             |                                                                   |               |                                                                          |
| Question                                                                                                                                                                                                                     |                      |                             |                                                                   |               |                                                                          |
| How has the 'Price of Excellence' changed? Can we predict the inflation-adjusted budget required to achieve a high IMDB rating (8.0+) based on the decade of release and genre?                                              |                      |                             |                                                                   |               |                                                                          |
|                                                                                                                                                                                                                              |                      |                             |                                                                   |               |                                                                          |
| Sampling                                                                                                                                                                                                                     |                      |                             |                                                                   |               |                                                                          |
| Yes, the data can be sampled. Since it is impractical to collect data on every movie ever made, we can sample the population. For this project, we are sampling by selecting only the Top 1000 most-rated on IMDB.com films. |                      |                             |                                                                   |               |                                                                          |


## Current Assignment

In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import time
from bs4 import BeautifulSoup
import pandas as pd
import re
import requests
import os
from dotenv import load_dotenv

### IMDB Movie ID Scraping
Collect top 1000 most voted films from IMDB.

Use Selenium to automatically click the "Load More" button 20 times (loading 50 movies per click) to reveal all 1,000 titles on the page.

Once loaded, use BeautifulSoup and Regex to parse the HTML and extract the unique IMDB ID (`tt0111161`) for each movie.

Save the list of IDs as a CSV file for use in the next steps.

In [ ]:
#Setting up chrome driver
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
try:
    #Using this specific url because it lists movies sorted by number of votes
    url = "https://www.imdb.com/search/title/?title_type=feature&sort=num_votes,desc"
    driver.get(url)
    wait = WebDriverWait(driver, 10)

    #Click the "Load More" button 20 times because 50 movies are loaded each time (50*20 = 1000)
    for i in range(20):
        try:
            #Wait for the button to be clickable
            load_more_button = wait.until(EC.element_to_be_clickable((By.CLASS_NAME, "ipc-see-more__button")))
            #Click the button
            driver.execute_script("arguments[0].click();", load_more_button)
            print(f"Clicked 'Load More' {i+1} times")
            #Pause to allow new content to load
            time.sleep(3) 
        except Exception as e:
            print("No more buttons found or error:", e)
            break
    #Extract the final HTML content after all the clicks
    page = driver.page_source
    print("HTML saved")
finally:
    #Close browser
    driver.quit()

Clicked 'Load More' 1 times
Clicked 'Load More' 2 times
Clicked 'Load More' 3 times
Clicked 'Load More' 4 times
Clicked 'Load More' 5 times
Clicked 'Load More' 6 times
Clicked 'Load More' 7 times
Clicked 'Load More' 8 times
Clicked 'Load More' 9 times
Clicked 'Load More' 10 times
Clicked 'Load More' 11 times
Clicked 'Load More' 12 times
Clicked 'Load More' 13 times
Clicked 'Load More' 14 times
Clicked 'Load More' 15 times
Clicked 'Load More' 16 times
Clicked 'Load More' 17 times
Clicked 'Load More' 18 times
Clicked 'Load More' 19 times
Clicked 'Load More' 20 times
HTML saved


In [ ]:
#Parse the HTML using BeautifulSoup
soup = BeautifulSoup(page, 'html.parser')
#Extract "ipc-title-link-wrapper", which holds the title links
movies = soup.find_all('a', class_='ipc-title-link-wrapper')

imdb_ids = []
for movie in movies:
    #Extract the ID from the link using regex
    imdb_ids.append(re.search(r'/title/(tt\d+)/', movie['href']).group(1))
    
imdb_ids[:5]

['tt0111161', 'tt0468569', 'tt1375666', 'tt0137523', 'tt0816692']

In [ ]:
#Convert the data to a pandas dataframe for easier saving
df = pd.DataFrame(imdb_ids, columns=['imdb_id'])
df.head()

,imdb_id
0,tt0111161
1,tt0468569
2,tt1375666
3,tt0137523
4,tt0816692


In [ ]:
#Save to a CSV file
df.to_csv('./data/raw/imdb_top_movies.csv', index=False)

### OMDB API Data Collection
Fetch detailed metadata for each movie using the OMDB API.

Use a set to track already-fetched IDs. This allows the script to be rerun safely (if the connection drops or the daily 1,000-request limit is reached) without having to begin from scratch again.

Iterate through the IMDB IDs, sending a request to OMDB for each one and appending the JSON response to our list.

The collected metadata is converted to a Pandas DataFrame and saved as a CSV file.

In [ ]:
#Load API key for OMDB
load_dotenv()
OMDB_API_KEY = os.getenv('OMDB_API_KEY')

#Init list to hold retrieved JSON data
omdb_data = []
#Init counter to track the number of fetches
fetch_count = 0

In [ ]:
#This code block should be run until the 2 numbers that are printed are equal, because the requests can often fail
#There is also a limit of 1000 requests per day, so I didn't use a for loop here (sometimes less than 1000 requests got tracked as 1000 and I ran into an infinite loop)

#Create a set of the IDs we already have so we don't call the API again for no reason
fetched_ids = {movie.get('imdbID') for movie in omdb_data}

#Loop through every IMDB id
for imdb_id in imdb_ids:
    #Skip IDs we already have
    if imdb_id in fetched_ids:
        continue
    try:
        #Request metadata from OMDB API
        response = requests.get(f"http://www.omdbapi.com/?i={imdb_id}&apikey={OMDB_API_KEY}")
        if response.status_code == 200:
            omdb_data.append(response.json())
            #Track success
            fetch_count += 1
        else:
            print(f"Failed to retrieve data for {imdb_id}")
    except Exception as e:
        #Catch network errors and request limits
        print(f"Error: {e}")

print(f"IMDB movie count: {len(imdb_ids)}")
print(f"Fetched movie count: {fetch_count}")

IMDB movie count: 1000
Fetched movie count: 1000


In [ ]:
#Convert to pandas dataframe because it's automatically converted from json to csv and easier to save
omdb_data_df = pd.DataFrame(omdb_data)
#Save file as CSV
omdb_data_df.to_csv('./data/raw/full_movie_data.csv', index=False)